<a href="https://colab.research.google.com/github/arildbn/bban4040/blob/main/martra-notebooks/4-1_bleu-evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div>
    <h1>Large Language Models Projects</a></h1>
    <h3>Apply and Implement Strategies for Large Language Models</h3>
    <h2>4.1-BLEU,  ROUGE and N-Grams. </h2>
    <h3>Evaluating translations with BLEU</h3>
    <p>by <b>Pere Martra</b></p>
</div>

In this notebook, we will use the BLEU metric to compare the quality of two different approaches for performing translations.

As my primary language is Spanish, I will translate a few lines from the beginning of this chapter from English to Spanish. My translations will be taken as the reference translations. In other words, they will be used as the basis upon which the quality of the automatic translations will be determined.



In [ ]:
#Sentences to Translate.
sentences = [
    "In the previous chapters, you've mainly seen how to work with OpenAI models, and you've had a very practical introduction to Hugging Face's open-source models, the use of embeddings, vector databases, and agents.",
    "These have been very practical chapters in which I've tried to gradually introduce concepts that have allowed you, or at least I hope so, to scale up your knowledge and start creating projects using the current technology stack of large language models."
    ]

In [ ]:
#Spanish Translation References.
reference_translations = [
    ["En los capítulos anteriores has visto mayoritariamente como trabajar con los modelos de OpenAI, y has tenido una introducción muy práctica a los modelos Open Source de Hugging Face, al uso de embeddings, las bases de datos vectoriales, los agentes."],
    ["Han sido capítulos muy prácticos en los que he intentado ir introduciendo conceptos que te han permitido, o eso espero, ir escalando en tus conocimientos y empezar a crear proyectos usando el stack tecnológico actual de los grandes modelos de lenguaje."]
    ]

We will perform the first translation using the NLLB model, a small model specialized in performing translations, which we will retrieve from Hugging Face.

In [ ]:
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from google.colab import userdata
import os

# Retrieve the secret and set it as an environment variable
try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    print("Successfully retrieved HF_TOKEN")
except Exception as e:
    print(f"Could not retrieve HF_TOKEN: {e}")

model_id = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id, token=hf_token)

When creating the pipeline, we pass the source language and the target language of the translation to it.

In [ ]:
# transformers 5.x removed the high-level 'translation' pipeline task, so we
# drive the NLLB model directly: set the source language on the tokenizer and
# force the target-language token at generation time.
def translate_nllb(text, src_lang="eng_Latn", tgt_lang="spa_Latn", max_length=400):
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt")
    tgt_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    generated = model.generate(**inputs, forced_bos_token_id=tgt_id, max_length=max_length)
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

In [ ]:
translations_nllb = []
for text in sentences:
    print("to translate: " + text)
    translated = translate_nllb(text)
    translations_nllb.append(translated)
    print(translated)

Now we have the translations stored in the list 'translations_nllb'.

In [ ]:
translations_nllb

##Create Translations with Google Traslator.

As a second source for translations, we will use the Google Translator API.

In [ ]:
# deep-translator is a reliable, synchronous Google Translate client (no API
# key needed). It replaces googletrans 4.x, whose async client returns a
# coroutine instead of text (so the translation was always skipped) and which
# frequently breaks with newer httpx.
%pip install -q deep-translator
try:
    from deep_translator import GoogleTranslator
    _GOOGLETRANS_OK = True
except Exception as e:
    print(f"deep-translator unavailable ({e}); the Google comparison will be skipped.")
    _GOOGLETRANS_OK = False

In [ ]:
# === portable-setup (bban4040) ===
# Secrets resolve from Colab "Secrets" (userdata) on Colab, or environment
# variables / a local .env file when running locally. Nothing is hardcoded.
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass
    value = os.environ.get(name, default)
    return value.strip() if isinstance(value, str) else value


_key = get_secret("OPENAI_API_KEY")
if _key:
    os.environ["OPENAI_API_KEY"] = _key


In [ ]:
# Translate English -> Spanish to match the reference translations.
translator_google = GoogleTranslator(source="en", target="es") if _GOOGLETRANS_OK else None

In [ ]:
translations_google = []
if _GOOGLETRANS_OK:
    for text in sentences:
        try:
            out = translator_google.translate(text)
            translations_google.append(out)
            print(out)
        except Exception as e:
            print(f"google translate failed ({e}); skipping Google comparison.")
            translations_google = []
            _GOOGLETRANS_OK = False
            break
else:
    print("Skipping Google translation (library unavailable).")

In this list, we have the translations created by Google.

In [ ]:
translations_google

## Evaluate translations with BLEU

We will use the BLEU implementation from the Evaluate library by Hugging Face.

In [ ]:
%pip install -q evaluate
import evaluate
bleu = evaluate.load('bleu')

In [ ]:
results_nllb = bleu.compute(predictions=translations_nllb, references=reference_translations)


To obtain the metrics, we pass the translated text and the reference text to the BLEU function.

Note that the translated text is a list of translations:
["Translation1", "Translation2"]

Whereas the reference texts are a list of lists of text. This allows for providing multiple references per translation:

[["reference1 Translation1", "reference2 Translation1"],
["reference2 Translation2", "reference2 Translation2"]]


In [ ]:
if translations_google:
    results_google = bleu.compute(predictions=translations_google, references=reference_translations)
    print(results_google)
else:
    print("No Google translations available - skipped Google BLEU.")

In [ ]:
print(results_nllb)

In [ ]:
if translations_google:
    print(results_google)
else:
    print("No Google BLEU (Google translation unavailable).")

It appears that the translation performed by the Google API is significantly better than the one performed by the NLLB model.